# Self-contained Unsloth Gemma 4 GGUF Chat

Upload this notebook to Google Drive / Colab and run the cell below. It clones upstream Unsloth, uses Unsloth's own `install_llama_prebuilt.py` to install `llama-server`, then opens llama.cpp chat with `unsloth/gemma-4-E4B-it-GGUF:UD-Q4_K_XL` already loaded.

Recommended Colab runtime: **GPU / T4**. Keep the final cell running while chatting.


In [ ]:
# One-cell Unsloth Gemma 4 GGUF chat launcher.
# If setup.sh failed before, this cell avoids the full Studio setup path and uses
# Unsloth's own llama.cpp prebuilt installer directly.
import base64
import os
import pathlib
import shutil
import subprocess

os.environ.setdefault('UNSLOTH_CHAT_MODEL', 'unsloth/gemma-4-E4B-it-GGUF:UD-Q4_K_XL')
os.environ.setdefault('UNSLOTH_CHAT_CTX', '4096')
os.environ.setdefault('UNSLOTH_CHAT_PORT', '8888')
# Optional example: os.environ['UNSLOTH_CHAT_EXTRA_ARGS'] = '--temp 0.7 --top-p 0.9'

UNSLOTH_REPO_DIR = pathlib.Path('/content/unsloth')
LLAMA_CPP_DIR = pathlib.Path.home() / '.unsloth' / 'llama.cpp'

if not (UNSLOTH_REPO_DIR / '.git').exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', 'main',
        'https://github.com/unslothai/unsloth.git', str(UNSLOTH_REPO_DIR),
    ], check=True)
else:
    print(f'Using existing checkout: {UNSLOTH_REPO_DIR}')

def has_gpu_tool():
    return any(shutil.which(name) for name in ('nvidia-smi', 'rocminfo', 'amd-smi', 'hipconfig', 'hipinfo'))

published_repo = os.environ.get('UNSLOTH_LLAMA_PUBLISHED_REPO')
if not published_repo:
    published_repo = 'unslothai/llama.cpp' if has_gpu_tool() else 'ggml-org/llama.cpp'

LLAMA_CPP_DIR.parent.mkdir(parents=True, exist_ok=True)
install_cmd = [
    'python', 'studio/install_llama_prebuilt.py',
    '--install-dir', str(LLAMA_CPP_DIR),
    '--llama-tag', os.environ.get('UNSLOTH_LLAMA_TAG', 'latest'),
    '--published-repo', published_repo,
    '--simple-policy',
]
print('Installing llama.cpp prebuilt via Unsloth installer:', published_repo)
result = subprocess.run(install_cmd, cwd=UNSLOTH_REPO_DIR)
if result.returncode != 0 and published_repo != 'ggml-org/llama.cpp':
    print('Primary prebuilt repo failed; retrying ggml-org/llama.cpp...')
    install_cmd[install_cmd.index('--published-repo') + 1] = 'ggml-org/llama.cpp'
    result = subprocess.run(install_cmd, cwd=UNSLOTH_REPO_DIR)
if result.returncode != 0:
    raise RuntimeError('llama.cpp prebuilt install failed. Scroll up for the installer log.')

LAUNCHER_B64 = (
    "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJMYXVuY2ggYSBwcmVsb2FkZWQgVW5zbG90aCBHZW1t"
    "YSA0IEdHVUYgY2hhdCBVSSB3aXRoIGxsYW1hLXNlcnZlci4KClRoZSBoZWF2eSBlbnZpcm9ubWVu"
    "dCBwcmVwYXJhdGlvbiBzdGF5cyBpbiBVbnNsb3RoJ3MgdXBzdHJlYW0gYGBzdHVkaW8vc2V0dXAu"
    "c2hgYC4KVGhpcyBmaWxlIG9ubHkgZmluZHMgdGhlIGluc3RhbGxlZCBgYGxsYW1hLXNlcnZlcmBg"
    "LCBzdGFydHMgaXQgd2l0aCB0aGUgZGVzaXJlZApIdWdnaW5nIEZhY2UgR0dVRiByZWZlcmVuY2Us"
    "IG9wZW5zIHRoZSBsbGFtYS5jcHAgY2hhdCBVSSwgYW5kIGtlZXBzIHRoZSBwcm9jZXNzCmFsaXZl"
    "IGZvciBDb2xhYi4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBv"
    "cnQgb3MKaW1wb3J0IHNobGV4CmltcG9ydCBzaHV0aWwKaW1wb3J0IHNpZ25hbAppbXBvcnQgc29j"
    "a2V0CmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCB0aW1lCmltcG9ydCB1cmxsaWIuZXJyb3IKaW1w"
    "b3J0IHVybGxpYi5yZXF1ZXN0CmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKREVGQVVMVF9NT0RF"
    "TF9SRUYgPSAidW5zbG90aC9nZW1tYS00LUU0Qi1pdC1HR1VGOlVELVE0X0tfWEwiCkRFRkFVTFRf"
    "UE9SVCA9IDg4ODgKCgpkZWYgX2hhc19ncHVfdG9vbCgpIC0+IGJvb2w6CiAgICBncHVfdG9vbHMg"
    "PSAoIm52aWRpYS1zbWkiLCAicm9jbWluZm8iLCAiYW1kLXNtaSIsICJoaXBjb25maWciLCAiaGlw"
    "aW5mbyIpCiAgICByZXR1cm4gYW55KHNodXRpbC53aGljaChuYW1lKSBmb3IgbmFtZSBpbiBncHVf"
    "dG9vbHMpCgoKZGVmIF9maW5kX2xsYW1hX3NlcnZlcigpIC0+IHN0cjoKICAgIGV4cGxpY2l0ID0g"
    "b3MuZ2V0ZW52KCJMTEFNQV9TRVJWRVJfUEFUSCIpCiAgICBpZiBleHBsaWNpdDoKICAgICAgICBw"
    "YXRoID0gUGF0aChleHBsaWNpdCkuZXhwYW5kdXNlcigpCiAgICAgICAgaWYgcGF0aC5pc19maWxl"
    "KCkgYW5kIG9zLmFjY2VzcyhwYXRoLCBvcy5YX09LKToKICAgICAgICAgICAgcmV0dXJuIHN0cihw"
    "YXRoKQogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiTExBTUFfU0VSVkVSX1BBVEgg"
    "aXMgbm90IGV4ZWN1dGFibGU6IHtwYXRofSIpCgogICAgY2FuZGlkYXRlcyA9IFsKICAgICAgICBQ"
    "YXRoLmhvbWUoKSAvICIudW5zbG90aCIgLyAibGxhbWEuY3BwIiAvICJidWlsZCIgLyAiYmluIiAv"
    "ICJsbGFtYS1zZXJ2ZXIiLAogICAgICAgIFBhdGguaG9tZSgpIC8gIi51bnNsb3RoIiAvICJsbGFt"
    "YS5jcHAiIC8gImxsYW1hLXNlcnZlciIsCiAgICAgICAgUGF0aC5ob21lKCkgLyAiLnVuc2xvdGgi"
    "IC8gInN0dWRpbyIgLyAibGxhbWEuY3BwIiAvICJidWlsZCIgLyAiYmluIiAvICJsbGFtYS1zZXJ2"
    "ZXIiLAogICAgICAgIFBhdGguaG9tZSgpIC8gIi51bnNsb3RoIiAvICJzdHVkaW8iIC8gImxsYW1h"
    "LmNwcCIgLyAibGxhbWEtc2VydmVyIiwKICAgIF0KICAgIGZvciBjYW5kaWRhdGUgaW4gY2FuZGlk"
    "YXRlczoKICAgICAgICBpZiBjYW5kaWRhdGUuaXNfZmlsZSgpIGFuZCBvcy5hY2Nlc3MoY2FuZGlk"
    "YXRlLCBvcy5YX09LKToKICAgICAgICAgICAgcmV0dXJuIHN0cihjYW5kaWRhdGUpCgogICAgb25f"
    "cGF0aCA9IHNodXRpbC53aGljaCgibGxhbWEtc2VydmVyIikKICAgIGlmIG9uX3BhdGg6CiAgICAg"
    "ICAgcmV0dXJuIG9uX3BhdGgKCiAgICBzZWFyY2hlZCA9ICJcbiIuam9pbihmIiAgLSB7cH0iIGZv"
    "ciBwIGluIGNhbmRpZGF0ZXMpCiAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcigKICAgICAgICAi"
    "Q291bGQgbm90IGZpbmQgbGxhbWEtc2VydmVyLiBSdW4gdW5zbG90aF9nZW1tYV9jaGF0L3NldHVw"
    "X2FuZF9sYXVuY2guc2ggZmlyc3QgIgogICAgICAgICJvciBzZXQgTExBTUFfU0VSVkVSX1BBVEgu"
    "IFNlYXJjaGVkOlxuIiArIHNlYXJjaGVkCiAgICApCgoKZGVmIF9wb3J0X2lzX29wZW4ocG9ydDog"
    "aW50KSAtPiBib29sOgogICAgd2l0aCBzb2NrZXQuc29ja2V0KHNvY2tldC5BRl9JTkVULCBzb2Nr"
    "ZXQuU09DS19TVFJFQU0pIGFzIHNvY2s6CiAgICAgICAgc29jay5zZXR0aW1lb3V0KDAuNSkKICAg"
    "ICAgICByZXR1cm4gc29jay5jb25uZWN0X2V4KCgiMTI3LjAuMC4xIiwgcG9ydCkpID09IDAKCgpk"
    "ZWYgX3dhaXRfZm9yX3NlcnZlcihwb3J0OiBpbnQsIHByb2Nlc3M6IHN1YnByb2Nlc3MuUG9wZW5b"
    "c3RyXSwgdGltZW91dF9zOiBpbnQgPSAxODAwKSAtPiBOb25lOgogICAgZGVhZGxpbmUgPSB0aW1l"
    "Lm1vbm90b25pYygpICsgdGltZW91dF9zCiAgICBsYXN0X3N0YXR1cyA9IDAuMAogICAgd2hpbGUg"
    "dGltZS5tb25vdG9uaWMoKSA8IGRlYWRsaW5lOgogICAgICAgIGlmIHByb2Nlc3MucG9sbCgpIGlz"
    "IG5vdCBOb25lOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJsbGFtYS1zZXJ2ZXIg"
    "ZXhpdGVkIGVhcmx5IHdpdGggY29kZSB7cHJvY2Vzcy5yZXR1cm5jb2RlfSIpCiAgICAgICAgaWYg"
    "X3BvcnRfaXNfb3Blbihwb3J0KToKICAgICAgICAgICAgZm9yIGVuZHBvaW50IGluICgiL2hlYWx0"
    "aCIsICIvdjEvbW9kZWxzIiwgIi8iKToKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAg"
    "ICAgICAgICB3aXRoIHVybGxpYi5yZXF1ZXN0LnVybG9wZW4oZiJodHRwOi8vMTI3LjAuMC4xOntw"
    "b3J0fXtlbmRwb2ludH0iLCB0aW1lb3V0PTIpOgogICAgICAgICAgICAgICAgICAgICAgICByZXR1"
    "cm4KICAgICAgICAgICAgICAgIGV4Y2VwdCB1cmxsaWIuZXJyb3IuSFRUUEVycm9yIGFzIGV4YzoK"
    "ICAgICAgICAgICAgICAgICAgICBpZiBleGMuY29kZSA8IDUwMDoKICAgICAgICAgICAgICAgICAg"
    "ICAgICAgcmV0dXJuCiAgICAgICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAg"
    "ICAgICAgICAgIHBhc3MKICAgICAgICBub3cgPSB0aW1lLm1vbm90b25pYygpCiAgICAgICAgaWYg"
    "bm93IC0gbGFzdF9zdGF0dXMgPiAxNToKICAgICAgICAgICAgcHJpbnQoIldhaXRpbmcgZm9yIEdl"
    "bW1hIDQgR0dVRiB0byBsb2FkLi4uIiwgZmx1c2g9VHJ1ZSkKICAgICAgICAgICAgbGFzdF9zdGF0"
    "dXMgPSBub3cKICAgICAgICB0aW1lLnNsZWVwKDEpCiAgICByYWlzZSBUaW1lb3V0RXJyb3IoZiJs"
    "bGFtYS1zZXJ2ZXIgZGlkIG5vdCBiZWNvbWUgcmVhZHkgd2l0aGluIHt0aW1lb3V0X3N9IHNlY29u"
    "ZHMiKQoKCmRlZiBfY29sYWJfcHJveHlfdXJsKHBvcnQ6IGludCkgLT4gc3RyOgogICAgZmFsbGJh"
    "Y2sgPSBmImh0dHA6Ly8xMjcuMC4wLjE6e3BvcnR9IgogICAgdHJ5OgogICAgICAgIGZyb20gZ29v"
    "Z2xlLmNvbGFiLm91dHB1dCBpbXBvcnQgZXZhbF9qcyAgIyB0eXBlOiBpZ25vcmUKICAgIGV4Y2Vw"
    "dCBFeGNlcHRpb246CiAgICAgICAgcmV0dXJuIGZhbGxiYWNrCgogICAgZm9yIF8gaW4gcmFuZ2Uo"
    "Myk6CiAgICAgICAgdHJ5OgogICAgICAgICAgICB1cmwgPSBldmFsX2pzKGYiZ29vZ2xlLmNvbGFi"
    "Lmtlcm5lbC5wcm94eVBvcnQoe3BvcnR9KSIsIHRpbWVvdXRfc2VjPTEwKQogICAgICAgICAgICBp"
    "ZiBpc2luc3RhbmNlKHVybCwgc3RyKSBhbmQgdXJsLnN0YXJ0c3dpdGgoImh0dHBzOi8vIik6CiAg"
    "ICAgICAgICAgICAgICByZXR1cm4gdXJsLnJzdHJpcCgiLyIpCiAgICAgICAgZXhjZXB0IEV4Y2Vw"
    "dGlvbjoKICAgICAgICAgICAgdGltZS5zbGVlcCgxKQogICAgcmV0dXJuIGZhbGxiYWNrCgoKZGVm"
    "IF9kaXNwbGF5X2NoYXQocG9ydDogaW50KSAtPiBOb25lOgogICAgdXJsID0gX2NvbGFiX3Byb3h5"
    "X3VybChwb3J0KQogICAgcHJpbnQoZiJcblVuc2xvdGggR2VtbWEgY2hhdCBpcyByZWFkeToge3Vy"
    "bH1cbiIsIGZsdXNoPVRydWUpCiAgICB0cnk6CiAgICAgICAgZnJvbSBJUHl0aG9uLmRpc3BsYXkg"
    "aW1wb3J0IEhUTUwsIGRpc3BsYXkgICMgdHlwZTogaWdub3JlCiAgICBleGNlcHQgRXhjZXB0aW9u"
    "OgogICAgICAgIHJldHVybgoKICAgIGRpc3BsYXkoSFRNTChmIiIiCjxkaXYgc3R5bGU9ImZvbnQt"
    "ZmFtaWx5OnN5c3RlbS11aSwtYXBwbGUtc3lzdGVtLHNhbnMtc2VyaWY7bWFyZ2luOjhweCAwO2Jv"
    "cmRlci1yYWRpdXM6MTJweDtvdmVyZmxvdzpoaWRkZW47Ym94LXNoYWRvdzowIDJweCAxNnB4IHJn"
    "YmEoMCwwLDAsLjE4KTsiPgogIDxkaXYgc3R5bGU9ImRpc3BsYXk6ZmxleDthbGlnbi1pdGVtczpj"
    "ZW50ZXI7Z2FwOjEwcHg7cGFkZGluZzoxMHB4IDE2cHg7YmFja2dyb3VuZDojMDAwO2NvbG9yOiNm"
    "ZmY7Ij4KICAgIDxzdHJvbmc+VW5zbG90aCBHZW1tYSA0IENoYXQ8L3N0cm9uZz4KICAgIDxhIGhy"
    "ZWY9Int1cmx9IiB0YXJnZXQ9Il9ibGFuayIgc3R5bGU9Im1hcmdpbi1sZWZ0OmF1dG87Y29sb3I6"
    "IzlhZTZiNDt0ZXh0LWRlY29yYXRpb246bm9uZTtmb250LXdlaWdodDo3MDA7Ij5PcGVuIGluIG5l"
    "dyB0YWI8L2E+CiAgPC9kaXY+CiAgPGlmcmFtZSBzcmM9Int1cmx9IiBzdHlsZT0id2lkdGg6MTAw"
    "JTtoZWlnaHQ6ODJ2aDttaW4taGVpZ2h0OjYyMHB4O2JvcmRlcjowO2Rpc3BsYXk6YmxvY2s7IiBh"
    "bGxvdz0iY2xpcGJvYXJkLXJlYWQ7IGNsaXBib2FyZC13cml0ZSI+PC9pZnJhbWU+CjwvZGl2Pgoi"
    "IiIpKQoKCmRlZiBfYnVpbGRfY29tbWFuZChiaW5hcnk6IHN0ciwgcG9ydDogaW50LCBtb2RlbF9y"
    "ZWY6IHN0cikgLT4gbGlzdFtzdHJdOgogICAgaG9zdCA9IG9zLmdldGVudigiVU5TTE9USF9DSEFU"
    "X0hPU1QiLCAiMC4wLjAuMCIpCiAgICBjdHhfc2l6ZSA9IG9zLmdldGVudigiVU5TTE9USF9DSEFU"
    "X0NUWCIsICI0MDk2IikKICAgIHBhcmFsbGVsID0gb3MuZ2V0ZW52KCJVTlNMT1RIX0NIQVRfUEFS"
    "QUxMRUwiLCAiMSIpCgogICAgY21kID0gWwogICAgICAgIGJpbmFyeSwKICAgICAgICAiLWhmIiwK"
    "ICAgICAgICBtb2RlbF9yZWYsCiAgICAgICAgIi0taG9zdCIsCiAgICAgICAgaG9zdCwKICAgICAg"
    "ICAiLS1wb3J0IiwKICAgICAgICBzdHIocG9ydCksCiAgICAgICAgIi1jIiwKICAgICAgICBjdHhf"
    "c2l6ZSwKICAgICAgICAiLS1wYXJhbGxlbCIsCiAgICAgICAgcGFyYWxsZWwsCiAgICAgICAgIi0t"
    "dGhyZWFkcyIsCiAgICAgICAgb3MuZ2V0ZW52KCJVTlNMT1RIX0NIQVRfVEhSRUFEUyIsICItMSIp"
    "LAogICAgICAgICItLWppbmphIiwKICAgICAgICAiLS1mbGFzaC1hdHRuIiwKICAgICAgICAib24i"
    "LAogICAgXQoKICAgIGlmIG9zLmdldGVudigiVU5TTE9USF9DSEFUX0dQVSIsICJhdXRvIikubG93"
    "ZXIoKSAhPSAib2ZmIiBhbmQgX2hhc19ncHVfdG9vbCgpOgogICAgICAgIGNtZC5leHRlbmQoWyIt"
    "bmdsIiwgIi0xIl0pCgogICAgZXh0cmEgPSBvcy5nZXRlbnYoIlVOU0xPVEhfQ0hBVF9FWFRSQV9B"
    "UkdTIiwgIiIpLnN0cmlwKCkKICAgIGlmIGV4dHJhOgogICAgICAgIGNtZC5leHRlbmQoc2hsZXgu"
    "c3BsaXQoZXh0cmEpKQogICAgcmV0dXJuIGNtZAoKCmRlZiBtYWluKCkgLT4gaW50OgogICAgbW9k"
    "ZWxfcmVmID0gb3MuZ2V0ZW52KCJVTlNMT1RIX0NIQVRfTU9ERUwiLCBERUZBVUxUX01PREVMX1JF"
    "RikKICAgIHBvcnQgPSBpbnQob3MuZ2V0ZW52KCJVTlNMT1RIX0NIQVRfUE9SVCIsIHN0cihERUZB"
    "VUxUX1BPUlQpKSkKICAgIHRpbWVvdXRfcyA9IGludChvcy5nZXRlbnYoIlVOU0xPVEhfQ0hBVF9M"
    "T0FEX1RJTUVPVVQiLCAiMTgwMCIpKQogICAgYmluYXJ5ID0gX2ZpbmRfbGxhbWFfc2VydmVyKCkK"
    "CiAgICBjbWQgPSBfYnVpbGRfY29tbWFuZChiaW5hcnksIHBvcnQsIG1vZGVsX3JlZikKICAgIHBy"
    "aW50KCJTdGFydGluZyBwcmVsb2FkZWQgY2hhdCBtb2RlbDoiLCBtb2RlbF9yZWYsIGZsdXNoPVRy"
    "dWUpCiAgICBwcmludCgiQ29tbWFuZDoiLCAiICIuam9pbihzaGxleC5xdW90ZShwYXJ0KSBmb3Ig"
    "cGFydCBpbiBjbWQpLCBmbHVzaD1UcnVlKQoKICAgIHByb2Nlc3MgPSBzdWJwcm9jZXNzLlBvcGVu"
    "KGNtZCwgdGV4dD1UcnVlKQoKICAgIGRlZiBfc3RvcChfc2lnbnVtOiBpbnQgfCBOb25lID0gTm9u"
    "ZSwgX2ZyYW1lOiBvYmplY3QgfCBOb25lID0gTm9uZSkgLT4gTm9uZToKICAgICAgICBpZiBwcm9j"
    "ZXNzLnBvbGwoKSBpcyBOb25lOgogICAgICAgICAgICBwcm9jZXNzLnRlcm1pbmF0ZSgpCiAgICAg"
    "ICAgICAgIHRyeToKICAgICAgICAgICAgICAgIHByb2Nlc3Mud2FpdCh0aW1lb3V0PTIwKQogICAg"
    "ICAgICAgICBleGNlcHQgc3VicHJvY2Vzcy5UaW1lb3V0RXhwaXJlZDoKICAgICAgICAgICAgICAg"
    "IHByb2Nlc3Mua2lsbCgpCiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgwKQoKICAgIHNpZ25hbC5z"
    "aWduYWwoc2lnbmFsLlNJR0lOVCwgX3N0b3ApCiAgICBzaWduYWwuc2lnbmFsKHNpZ25hbC5TSUdU"
    "RVJNLCBfc3RvcCkKCiAgICBfd2FpdF9mb3Jfc2VydmVyKHBvcnQsIHByb2Nlc3MsIHRpbWVvdXRf"
    "cz10aW1lb3V0X3MpCiAgICBfZGlzcGxheV9jaGF0KHBvcnQpCgogICAgcHJpbnQoIktlZXAgdGhp"
    "cyBjZWxsL3Byb2Nlc3MgcnVubmluZyB0byBrZWVwIHRoZSBjaGF0IHNlcnZlciBhbGl2ZS4iLCBm"
    "bHVzaD1UcnVlKQogICAgd2hpbGUgcHJvY2Vzcy5wb2xsKCkgaXMgTm9uZToKICAgICAgICB0aW1l"
    "LnNsZWVwKDMwMCkKICAgICAgICBwcmludCgiPSIsIGVuZD0iIiwgZmx1c2g9VHJ1ZSkKICAgIHJl"
    "dHVybiBpbnQocHJvY2Vzcy5yZXR1cm5jb2RlIG9yIDApCgoKaWYgX19uYW1lX18gPT0gIl9fbWFp"
    "bl9fIjoKICAgIHJhaXNlIFN5c3RlbUV4aXQobWFpbigpKQo="
)
launcher = base64.b64decode(LAUNCHER_B64).decode('utf-8')
launcher_path = pathlib.Path('/content/unsloth_gemma_chat_launch.py')
launcher_path.write_text(launcher, encoding='utf-8')
subprocess.run(['python', str(launcher_path)], check=True)
